In [ ]:
# ============================================================
# 🎧 Customer Support Knowledge Base using Vector Databases
# Comparative Analysis of Qdrant and pgvector
# ============================================================

# Objective:
# This project compares two vector databases, Qdrant and pgvector,
# by implementing the same Customer Support Knowledge Base use case
# using semantic search on customer queries and support issues.

# Embedding Model:
# sentence-transformers/all-MiniLM-L6-v2

# Vector Databases Compared:
# 1. Qdrant
# 2. pgvector (PostgreSQL Extension)

# Platform:
# Google Colab (CPU Runtime)

# Workflow:
# 1. Create a customer support knowledge base dataset.
# 2. Generate embeddings using Sentence Transformers.
# 3. Store embeddings in Qdrant.
# 4. Store the same embeddings in pgvector.
# 5. Search relevant support articles using semantic similarity.
# 6. Compare search results and performance.


In [ ]:
# ============================================================
# CELL 2 : Install Required Python Libraries
# ============================================================

!pip -q install qdrant-client sentence-transformers pandas tabulate matplotlib psycopg2-binary


In [ ]:
# ============================================================
# CELL 3 : Import Libraries
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

print("✅ Libraries imported successfully.")


In [ ]:
# ============================================================
# CELL 4 : Generate 100 Customer Support Records
#           (25 Topics × 4 Variations)
# ============================================================

import pandas as pd

# Base customer-support topics with category and issue description
base_support_topics = [
    ("Payment Failed", "Payments",
     "payment was declined, transaction failed, unable to complete payment"),

    ("Payment Deducted but Order Failed", "Payments",
     "money was deducted but the order was not successfully placed"),

    ("Refund Not Received", "Refunds",
     "refund was approved but the money has not been received"),

    ("Wrong Refund Amount", "Refunds",
     "refund amount received is different from the expected amount"),

    ("Order Not Delivered", "Order & Delivery",
     "order has not arrived even though the expected delivery date has passed"),

    ("Late Delivery", "Order & Delivery",
     "order delivery is delayed and taking longer than expected"),

    ("Order Tracking Issue", "Order & Delivery",
     "unable to track the order or tracking information is not updating"),

    ("Wrong Item Received", "Orders",
     "received a different product from the one that was ordered"),

    ("Damaged Item Received", "Orders",
     "product arrived damaged, broken, or unusable"),

    ("Cancel Order", "Order Cancellation",
     "customer wants to cancel an order before it is delivered"),

    ("Unable to Cancel Order", "Order Cancellation",
     "customer cannot cancel an order through the application"),

    ("Account Login Problem", "Account & Login",
     "unable to log into the account using the correct credentials"),

    ("Forgot Password", "Account & Login",
     "customer forgot the account password and needs to reset it"),

    ("Account Locked", "Account & Login",
     "account has been locked after multiple unsuccessful login attempts"),

    ("Change Phone Number", "Account & Login",
     "customer wants to update the phone number linked to the account"),

    ("Change Email Address", "Account & Login",
     "customer wants to change the email address associated with the account"),

    ("Suspicious Account Activity", "Security",
     "customer noticed unfamiliar activity or transactions on the account"),

    ("Unauthorized Transaction", "Security",
     "customer reports a transaction that they did not authorize"),

    ("Coupon Not Working", "Offers & Discounts",
     "discount coupon is not being applied during checkout"),

    ("Discount Missing", "Offers & Discounts",
     "expected discount is missing from the order total"),

    ("Subscription Charge", "Billing",
     "customer was charged for a subscription unexpectedly"),

    ("Duplicate Charge", "Billing",
     "customer was charged more than once for the same transaction"),

    ("App Not Working", "Technical Issues",
     "application crashes, freezes, or does not work correctly"),

    ("Website Error", "Technical Issues",
     "website displays an error while customer is trying to complete an action"),

    ("Product Information", "General Support",
     "customer needs information about product availability, features, or specifications")
]

# Four different variations for each support topic
variations = [
    "Customer reports {}.",
    "Customer is experiencing {} and needs assistance.",
    "Customer complains about {}.",
    "Customer is facing an issue where {}."
]

support_data = []
record_id = 1

for issue, category, description in base_support_topics:
    for variation in variations:
        support_data.append({
            "id": record_id,
            "issue": issue,
            "description": variation.format(description),
            "category": category
        })
        record_id += 1

# Create DataFrame
df = pd.DataFrame(support_data)

print("✅ Customer Support Dataset Created Successfully!")
print(f"Total Records: {len(df)}")
print(f"Unique Support Topics: {df['issue'].nunique()}")

display(df.head(10))


In [ ]:
# ============================================================
# CELL 5 : Generate Customer Support Embeddings
# ============================================================

print("Loading Sentence Transformer model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Generating embeddings...")

# Combine issue name and description for richer semantic search
search_text = (
    df["issue"] + ". " +
    df["description"]
)

embeddings = embedding_model.encode(
    search_text.tolist(),
    show_progress_bar=True
)

# Store embeddings inside dataframe
df["embedding"] = embeddings.tolist()

print(f"Embedding Dimension : {len(embeddings[0])}")
print(f"Total Embeddings Generated : {len(embeddings)}")


In [ ]:
# ============================================================
# CELL 6 : Initialize Qdrant Vector Database
# ============================================================

# Qdrant runs completely in memory for this project.
# No Docker or server is required in Google Colab.

qdrant_client = QdrantClient(":memory:")

# Collection for our customer support knowledge base
collection_name = "customer_support"

print("✅ Qdrant initialized successfully.")


In [ ]:
# ============================================================
# CELL 7 : Create Qdrant Collection
# ============================================================

# A collection is similar to a SQL table.
# It stores vectors of dimension 384 using cosine similarity.

qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print(f"✅ Collection '{collection_name}' created.")


In [ ]:
# ============================================================
# CELL 8 : Upload Support Tickets into Qdrant (100 Records)
# ============================================================

points = []

# Convert each support record into a Qdrant point
for _, row in df.iterrows():

    point = PointStruct(
        # Use the dataset ID (1 to 100)
        id=int(row["id"]),

        # 384-dimensional embedding vector
        vector=row["embedding"],

        # Metadata stored along with the vector
        payload={
            "issue": row["issue"],
            "description": row["description"],
            "category": row["category"]
        }
    )

    points.append(point)

# Upload all vectors into the collection
qdrant_client.upsert(
    collection_name=collection_name,
    points=points
)

# Verify upload
collection_info = qdrant_client.get_collection(collection_name)

print(f"✅ Uploaded {len(points)} ticket vectors into Qdrant.")
print(f"📊 Total vectors stored in Qdrant: {collection_info.points_count}")


In [ ]:
# ============================================================
# CELL 9 : Verify Collection Information
# ============================================================

collection_info = qdrant_client.get_collection(collection_name)

print("Collection Name :", collection_name)
print("Vectors Stored  :", collection_info.points_count)
print("Distance Metric :", collection_info.config.params.vectors.distance)


In [ ]:
# ============================================================
# CELL 10 : Search Customer Support Issues using Qdrant
# ============================================================

def search_support_qdrant(user_query, top_k=5):
    """
    Search customer-support issues using semantic similarity in Qdrant.
    """

    # Convert customer query into a 384-dimensional embedding
    query_vector = embedding_model.encode(user_query).tolist()

    # Measure search time
    start_time = time.perf_counter()

    # Search for the most similar vectors in Qdrant
    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k
    )

    end_time = time.perf_counter()

    # Convert search time to milliseconds
    search_time = (end_time - start_time) * 1000

    output = []

    # Extract information from Qdrant results
    for point in results.points:
        output.append({
            "Issue": point.payload["issue"],
            "Category": point.payload["category"],
            "Similarity Score": round(point.score, 4)
        })

    # Convert results into a DataFrame
    result_df = pd.DataFrame(output)

    return result_df, search_time


# ---------------- Example Search ----------------

customer_query = "My payment was charged but my order was not placed"

results, qdrant_time = search_support_qdrant(customer_query)

print("Customer Query:")
print(customer_query)

print(f"\nQdrant Search Time: {qdrant_time:.2f} ms\n")

display(results)


## Part 2 — pgvector (PostgreSQL) Implementation

In [ ]:
# ============================================================
# CELL 11 : Install PostgreSQL + pgvector (Run Once)
# ============================================================

# Install PostgreSQL
!apt-get update -qq
!apt-get install -y postgresql postgresql-common postgresql-contrib postgresql-server-dev-14 build-essential git

# Download pgvector only if it doesn't exist
import os

if not os.path.exists("pgvector"):
    !git clone https://github.com/pgvector/pgvector.git

# Build and install pgvector
%cd pgvector
!make
!make install
%cd ..

# Start PostgreSQL
!pg_ctlcluster 14 main start

# Set postgres password
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"

# Create database (ignore if already exists)
!sudo -u postgres psql -c "CREATE DATABASE customer_support_db;" || echo "customer_support_db already exists"

# Restart PostgreSQL so pgvector is detected
!pg_ctlcluster 14 main restart

# Enable pgvector extension
!sudo -u postgres psql -d customer_support_db -c "CREATE EXTENSION IF NOT EXISTS vector;"

print("✅ PostgreSQL + pgvector setup completed.")


In [ ]:
# ============================================================
# CELL 12 : Connect to PostgreSQL
# ============================================================

import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="customer_support_db",
    user="postgres",
    password="postgres",
    port=5432
)

cursor = conn.cursor()

print("✅ Connected to PostgreSQL successfully.")


In [ ]:
# ============================================================
# CELL 13 : Create Table with VECTOR Column
# ============================================================

cursor.execute("DROP TABLE IF EXISTS support_tickets;")

cursor.execute("""
CREATE TABLE support_tickets(
    id INTEGER PRIMARY KEY,
    issue TEXT,
    description TEXT,
    category TEXT,
    embedding VECTOR(384)
);
""")

conn.commit()

print("✅ support_tickets table created.")


In [ ]:
# ============================================================
# CELL 14 : Insert 100 Embeddings into pgvector (Final Version)
# ============================================================

# Reset any failed transaction
conn.rollback()

# Remove existing records
cursor.execute("TRUNCATE TABLE support_tickets;")
conn.commit()

# Insert all records
for _, row in df.iterrows():

    # Convert embedding list into PostgreSQL vector format
    vector_str = "[" + ",".join(map(str, row["embedding"])) + "]"

    cursor.execute("""
        INSERT INTO support_tickets
        (id, issue, description, category, embedding)
        VALUES (%s, %s, %s, %s, %s::vector);
    """,
    (
        int(row["id"]),
        row["issue"],
        row["description"],
        row["category"],
        vector_str
    ))

# Save all inserts
conn.commit()

# Verify insertion
cursor.execute("SELECT COUNT(*) FROM support_tickets;")
count = cursor.fetchone()[0]

print(f"✅ Successfully inserted {count} records into pgvector.")


In [ ]:
cursor.execute("SELECT COUNT(*) FROM support_tickets;")
count = cursor.fetchone()[0]

print("Records Stored:", count)


In [ ]:
# ============================================================
# CELL 15 : Customer Support Search using pgvector
# ============================================================

def search_issue_pgvector(user_query, top_k=5):

    # Convert query into embedding
    query_embedding = embedding_model.encode(user_query).tolist()

    # Convert embedding into PostgreSQL vector format
    vector_str = "[" + ",".join(map(str, query_embedding)) + "]"

    start_time = time.perf_counter()

    cursor.execute("""
        SELECT
            issue,
            category,
            ROUND((1 - (embedding <=> %s::vector))::numeric, 4) AS similarity_score
        FROM support_tickets
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """,
    (vector_str, vector_str, top_k))

    rows = cursor.fetchall()

    end_time = time.perf_counter()
    search_time = (end_time - start_time) * 1000

    results_df = pd.DataFrame(
        rows,
        columns=["Issue", "Category", "Similarity Score"]
    )

    return results_df, search_time


# ---------------- Test the Function ----------------

customer_query = "payment got deducted twice for the same order"

pg_results, pg_time = search_issue_pgvector(customer_query)

print("Customer Query:")
print(customer_query)

print(f"\npgvector Search Time: {pg_time:.2f} ms\n")

display(pg_results)


In [ ]:
# ============================================================
# CELL 16 : Compare Qdrant and pgvector
# ============================================================

# Same test queries are used for BOTH vector databases.
test_queries = [
    "unable to login, account locked",
    "order not delivered yet",
    "app crashes every time I open it",
    "refund not received for cancelled order",
    "coupon code not working at checkout"
]

comparison_results = []

for query in test_queries:

    # ---------- Qdrant Search ----------
    qdrant_result, qdrant_time = search_support_qdrant(query)

    # ---------- pgvector Search ----------
    pg_result, pg_time = search_issue_pgvector(query)

    comparison_results.append({
        "Customer Query": query,
        "Qdrant Prediction": qdrant_result.iloc[0]["Issue"],
        "pgvector Prediction": pg_result.iloc[0]["Issue"],
        "Qdrant Time (ms)": round(qdrant_time, 2),
        "pgvector Time (ms)": round(pg_time, 2)
    })

comparison_df = pd.DataFrame(comparison_results)

print("✅ Comparison completed successfully.\n")

display(comparison_df)


In [ ]:
# ============================================================
# CELL 17 : Performance Comparison Chart
# ============================================================

# Calculate average search time
avg_qdrant = comparison_df["Qdrant Time (ms)"].mean()
avg_pgvector = comparison_df["pgvector Time (ms)"].mean()

print(f"Average Qdrant Search Time : {avg_qdrant:.2f} ms")
print(f"Average pgvector Search Time : {avg_pgvector:.2f} ms")

# Plot bar chart
plt.figure(figsize=(6,4))

plt.bar(
    ["Qdrant", "pgvector"],
    [avg_qdrant, avg_pgvector]
)

plt.title("Average Search Time Comparison")
plt.ylabel("Time (milliseconds)")
plt.xlabel("Vector Database")

plt.grid(axis="y", alpha=0.3)

plt.show()


In [ ]:
# ============================================================
# CELL 18 : Evaluation Summary
# ============================================================

evaluation_df = pd.DataFrame({
    "Evaluation Metric": [
        "Use Case",
        "Embedding Model",
        "Similarity Metric",
        "Metadata Support",
        "Database Type",
        "Average Search Time"
    ],

    "Qdrant": [
        "Customer Support Knowledge Base",
        "all-MiniLM-L6-v2",
        "Cosine Similarity",
        "Payload Metadata Filtering",
        "Dedicated Vector Database",
        f"{avg_qdrant:.2f} ms"
    ],

    "pgvector": [
        "Customer Support Knowledge Base",
        "all-MiniLM-L6-v2",
        "Cosine Similarity",
        "SQL WHERE + Vector Search",
        "PostgreSQL Extension",
        f"{avg_pgvector:.2f} ms"
    ]
})

print("========== FINAL EVALUATION ==========\n")
display(evaluation_df)

print("\n========== PROJECT CONCLUSION ==========\n")

print("""
1. Both Qdrant and pgvector successfully retrieved matching support tickets
   using semantic similarity search.

2. Qdrant was easier to configure and use for AI-native vector search.

3. pgvector integrates semantic search directly inside PostgreSQL,
   making it suitable for support systems that already store ticket
   data relationally.

4. Since both systems used the same embeddings and dataset,
   the comparison is fair and demonstrates how different vector databases
   solve the same semantic retrieval problem for a customer support
   knowledge base.
""")
